In [ ]:
from abc import ABC, abstractmethod

import numpy as np

In [ ]:
class BaseForecastModel(ABC):
    # Params kept as int
    INT_PARAMS = ()
    DEFAULT_SPACE = {}
    N_GRID_POINTS = 3

    @classmethod
    def _cast_int_params(cls, params):
        return {
            k: (int(v) if k in cls.INT_PARAMS and v is not None else v)
            for k, v in params.items()
        }

    def fit(self, X_train, y_train):
        self.estimator.fit(X_train, y_train)
        return self

    def predict(self, X):
        return self.estimator.predict(X)

    def reset_forecast(self):
        # No-op for stateless models
        pass

    def advance(self, X_new, y_new):
        # No-op for stateless models
        pass

    @classmethod
    def get_search_space(cls, config_ranges, n_grid_points=None):
        # Two values are a [lo, hi] range, anything else is a literal list
        ranges = {**cls.DEFAULT_SPACE, **(config_ranges or {})}
        points = n_grid_points or cls.N_GRID_POINTS
        grid = {}
        for name, spec in ranges.items():
            if len(spec) != 2:
                grid[name] = list(spec)
                continue
            lo, hi = spec
            values = np.linspace(lo, hi, points)
            if name in cls.INT_PARAMS:
                grid[name] = sorted(set(int(v) for v in values))
            else:
                grid[name] = list(values)
        return grid

    @classmethod
    @abstractmethod
    def build(cls, params):
        """Instantiate the wrapped estimator from a param set."""